# Day 2: Data Types and Distributions

## Objectives
- Understand numerical, categorical, and boolean data types
- Distinguish continuous vs discrete and nominal vs ordinal variables
- Detect and fix incorrect pandas dtypes
- Visualize numerical and categorical distributions
- Optimize memory usage using appropriate dtypes
- Understand semantic type errors in ML pipelines

**Dataset:** Airbnb NYC Open Data (Kaggle)


## Theory

### Numerical Data
- Continuous: price, age, temperature
- Discrete: number of bedrooms, number of reviews

### Categorical Data
- Nominal: neighborhood, city, country
- Ordinal: education level, star ratings

### Boolean Data
- True/False values

### Why Dtypes Matter
Wrong data types can silently break preprocessing pipelines and cause poor model performance.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme()


## Load Dataset

In [ ]:
# Update the file path as needed
df = pd.read_csv('AB_NYC_2019.csv')

print('Shape:', df.shape)
df.head()


## Audit Data Types

In [ ]:
df.info()

print('\nData Types:')
print(df.dtypes)


## Memory Usage Before Optimization

In [ ]:
memory_before = df.memory_usage(deep=True).sum() / 1024**2
print(f'Memory Usage Before Optimization: {memory_before:.2f} MB')


## Identify Potentially Incorrect Types

In [ ]:
# Example columns that are often categorical despite looking numeric
possible_categorical = ['neighbourhood_group', 'neighbourhood', 'room_type']

for col in possible_categorical:
    if col in df.columns:
        print(col, df[col].dtype)


## Fix Data Types Explicitly

In [ ]:
dtype_fixes = {
    'neighbourhood_group': 'category',
    'neighbourhood': 'category',
    'room_type': 'category'
}

for col, dtype in dtype_fixes.items():
    if col in df.columns:
        df[col] = df[col].astype(dtype)

df.dtypes.head(20)


## Numerical Columns Distribution

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in numerical_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()


## Categorical Columns Distribution

In [ ]:
categorical_cols = df.select_dtypes(include=['category', 'object']).columns

for col in categorical_cols:
    plt.figure(figsize=(8,4))
    df[col].value_counts().head(15).plot(kind='bar')
    plt.title(f'Top Categories in {col}')
    plt.ylabel('Count')
    plt.show()


## Memory Usage After Optimization

In [ ]:
memory_after = df.memory_usage(deep=True).sum() / 1024**2

print(f'Memory Usage Before: {memory_before:.2f} MB')
print(f'Memory Usage After : {memory_after:.2f} MB')
print(f'Reduction         : {(memory_before-memory_after):.2f} MB')


## Semantic Type Error Example

Zip codes should be categorical, not numerical.

Bad:
```python
df['zipcode'] = df['zipcode'].astype(int)
```

Good:
```python
df['zipcode'] = df['zipcode'].astype('category')
```


## Interview Question

**Q:** You have a star-rating column (1–5). A colleague encodes it as numerical. What's the problem?

### Answer
Star rating is ordinal, not truly numerical.

- Linear models may assume equal spacing between ratings.
- Ordinal encoding preserves ranking.
- One-hot encoding removes artificial numeric relationships.
- Tree-based models generally handle ordinal encoding well.


## ML Spotlight: Pandas 2.0 Arrow Backend

```python
df = pd.read_csv(
    'AB_NYC_2019.csv',
    dtype_backend='pyarrow'
)
```

Benefits:
- Reduced memory usage
- Faster operations
- Better scalability for production ML pipelines
